# Step C review — matches & misses

Regenerate — **do not hand-edit** the `.ipynb` JSON:

```bash
uv run python scripts/generate_notebooks.py --name step_c_review
```

**Display:** plain pandas + `walk_details` (works in Cursor). No itables/widgets required.

Compare hand gold `step_c` to predictions. Edit **CONFIG**, re-run load / filter cells.

CLI: `uv run python scripts/export_step_c_review.py`


## Config

Edit this cell, then re-run the cells below.


In [ ]:
from pathlib import Path
from dataclasses import replace

from data_io import resolve
from trifecta_annotation.step_c_review import (
    StepCReviewConfig,
    load_review_frame,
    review_summary,
    save_review_frame,
    show_review,
    walk_details,
)

SCRATCH = Path(resolve('trifecta_gold')).parent
EVAL = Path(resolve('eval_reports'))

# --- knobs (change for future frames / slices) ---
CONFIG = StepCReviewConfig(
    gold_path=None,
    predictions_path=SCRATCH / 'gold_predictions.jsonl',
    output_path=EVAL / 'step_c_review.csv',
    frames=(),  # e.g. ('COOKING_CREATION',)
    tiers=(),  # e.g. ('joint', 'partial_strong', 'miss')
    fewshot_only=False,
    fewshot_levels=('yes', 'maybe'),
    min_hits=0,
    frame_ok_only=False,
)

CONFIG


## Load

In [ ]:
review = load_review_frame(CONFIG)
review_summary(review)


## Compact table

Pandas view (scrollable). Filter with the next cell if needed.


In [ ]:
show_review(review, CONFIG)  # prefer_itables=True only in classic Jupyter


## Filter with pandas (no widgets)

Change the boolean mask, re-run.


In [ ]:
# Examples — uncomment one:
# slice_ = review[review['tier'].isin(['joint', 'partial_strong'])]
# slice_ = review[review['gold_frame'] == 'COOKING_CREATION']
# slice_ = review[review['fewshot_candidate'].astype(str).str.len() > 0]
slice_ = review.copy()

print(review_summary(slice_))
show_review(slice_, CONFIG)


## Read rows in detail (gold vs pred)

Step through with `start` / `n`. Longer predictions are often fine — exact-match scores understate quality when pred ⊃ gold.


In [ ]:
walk_details(slice_, start=0, n=5)
# walk_details(slice_, start=5, n=5)  # next page


## Few-shot shortlist + save

In [ ]:
FEWSHOT = replace(
    CONFIG,
    fewshot_only=True,
    # frames=('COOKING_CREATION',),
    output_path=EVAL / 'step_c_fewshot_candidates.csv',
)
few = load_review_frame(FEWSHOT)
print(review_summary(few))
walk_details(few, start=0, n=10)

path_all = save_review_frame(review, CONFIG)
path_few = save_review_frame(few, FEWSHOT)
print('all →', path_all)
print('fewshot →', path_few)


## How to use this

1. `walk_details` — compare GOLD vs PRED side by side.
2. If you prefer longer PRED text, treat exact-match misses as **style**, not necessarily wrong — note for soft scoring later.
3. Pick record_ids for few-shots (or say “all yes”) and wire into Step C prompts.
4. Optional: open the saved CSV in Cursor for search/notes.
